# Pilot — match LLM samples with 2 similar HUMAN samples (stratified subset)

Reads a pilot generation file (`v2/data/interim/llm-tests/pilot_*.jsonl`), builds a **small stratified subset** of LLM rows (default **20**), then for each selected row finds **two** most similar human examples matched by:

- `scenario_family`, `channel`, `fraudness` (hard constraints)
- same `subtype` when available on the human side (preferred)
- same `length_bin` (LLM `actual_bin` ↔ human `length_bin`) (preferred)
- closest `token_length` (absolute difference)

**Stratification:** ensures every `scenario_family` appears at least once, then adds rows to diversify `(subtype, actual_bin)` strata, then fills to `N_STRATIFIED` if needed. Adjust `N_STRATIFIED` / `RANDOM_STATE_STRAT` in the setup cell.

Writes a markdown report to `v2/data/interim/llm-tests/match_s{N}_{YYYYMMDD_HHMMSS}.md` (short name), same visual style as older `pilot_human_llm_examples_*.md` reports, with **two** HUMAN blocks per included LLM example (not the full pilot).

Human sources used:
- fraud email: `v2/data/interim/gathered/nazario_prepared.jsonl`
- fraud 419 email: `v2/data/interim/gathered/nigerian_fraud_prepared.jsonl`
- fraud SMS: `v2/data/interim/gathered/smishtank_prepared.jsonl` + `mendeley_smishing_prepared.jsonl`
- ham email: `v2/data/interim/annotated/enron_ham_annotated.jsonl` + `spamassassin_ham_annotated.jsonl`
- ham SMS: `v2/data/interim/annotated/sms_ham_annotated.jsonl`

Length bins come from `v2/config.py` (token thresholds).

In [31]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
from __future__ import annotations

import json
import sys
import datetime
from pathlib import Path

import pandas as pd

BASE = Path("/Users/askar/projects/antifraud-deepfake-detection/v2")
OUT_DIR = BASE / "data" / "interim" / "llm-tests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use latest pilot_*.jsonl by mtime so reports are not accidentally built from an old run.
# Override: set PILOT_PATH_OVERRIDE = OUT_DIR / "pilot_<stem>.jsonl"
PILOT_PATH_OVERRIDE = None  # Path | None
if PILOT_PATH_OVERRIDE is not None:
    PILOT_PATH = PILOT_PATH_OVERRIDE
else:
    _pilots = sorted(OUT_DIR.glob("pilot_*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)
    assert _pilots, f"No pilot_*.jsonl in {OUT_DIR}"
    PILOT_PATH = _pilots[0]

# v2/config.py
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin

print("BASE:", BASE)
print("PILOT:", PILOT_PATH)
assert PILOT_PATH.exists(), f"Pilot file not found: {PILOT_PATH}"

llm = pd.read_json(PILOT_PATH, lines=True)
print("LLM rows (full pilot):", len(llm))

# ── Stratified subset size (visual comparison, not full pilot) ─────────────
N_STRATIFIED = 20
RANDOM_STATE_STRAT = 42


def stratified_llm_sample(df: pd.DataFrame, n: int, random_state: int) -> pd.DataFrame:
    """Pick n pilot rows: all scenario_families covered, then diversify (subtype, actual_bin)."""
    work = df.copy()
    if "gen_id" not in work.columns:
        work["gen_id"] = work.index.astype(str)

    fams = sorted(work["scenario_family"].dropna().unique())
    if n < len(fams):
        raise ValueError(f"N_STRATIFIED ({n}) must be >= number of families ({len(fams)})")

    picked: list[pd.Series] = []
    used: set[str] = set()

    def add_row(row: pd.Series) -> bool:
        gid = str(row["gen_id"])
        if gid in used:
            return False
        used.add(gid)
        picked.append(row)
        return True

    # Phase 1: at least one row per scenario_family
    for i, fam in enumerate(fams):
        sub = work[work["scenario_family"] == fam]
        if len(sub) == 0:
            continue
        row = sub.sample(1, random_state=random_state + 17 * i).iloc[0]
        add_row(row)

    # Phase 2: add one row per distinct (family, subtype, actual_bin) stratum (shuffled)
    rest = work[~work["gen_id"].astype(str).isin(used)].copy()
    rest["_stratum"] = (
        rest["scenario_family"].astype(str)
        + "||"
        + rest["subtype"].astype(str)
        + "||"
        + rest["actual_bin"].astype(str)
    )
    stratum_keys = (
        pd.Series(rest["_stratum"].unique())
        .sample(frac=1.0, random_state=random_state)
        .tolist()
    )
    k = 0
    for key in stratum_keys:
        if len(picked) >= n:
            break
        sub = rest[rest["_stratum"] == key]
        sub = sub[~sub["gen_id"].astype(str).isin(used)]
        if len(sub) == 0:
            continue
        row = sub.sample(1, random_state=random_state + 97 + k).iloc[0]
        if add_row(row):
            k += 1

    # Phase 3: fill to n from remaining pool
    pool = work[~work["gen_id"].astype(str).isin(used)]
    fill_seed = random_state + 10_000
    while len(picked) < n and len(pool) > 0:
        row = pool.sample(1, random_state=fill_seed).iloc[0]
        fill_seed += 1
        if not add_row(row):
            pool = pool.drop(index=row.name)
            if len(pool) == 0:
                break

    out = pd.DataFrame(picked)
    return out.sort_values(
        ["scenario_family", "subtype", "token_count"],
        kind="stable",
    ).reset_index(drop=True)


llm_to_match = stratified_llm_sample(llm, n=N_STRATIFIED, random_state=RANDOM_STATE_STRAT)
print("Stratified subset rows:", len(llm_to_match), "(target", N_STRATIFIED, ")")
print("Coverage by family:\n", llm_to_match["scenario_family"].value_counts().sort_index().to_string())
print("\nStrata (family || subtype || actual_bin) in subset:")
print(llm_to_match.assign(_s=llm_to_match["scenario_family"].astype(str) + "||" + llm_to_match["subtype"].astype(str) + "||" + llm_to_match["actual_bin"].astype(str))["_s"].value_counts().sort_index().to_string())

llm_to_match.head(3)

BASE: /Users/askar/projects/antifraud-deepfake-detection/v2
PILOT: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-tests/pilot_20260408_005559_455333_55017f6f.jsonl
LLM rows (full pilot): 115
Stratified subset rows: 20 (target 20 )
Coverage by family:
 scenario_family
advance_fee_scam_email    3
fraud_sms_deceptive       5
legitimate_email          3
legitimate_sms            1
phishing_email            8

Strata (family || subtype || actual_bin) in subset:
_s
advance_fee_scam_email||diplomatic_package_or_fund_release||medium    1
advance_fee_scam_email||inheritance_or_estate_transfer||medium        2
fraud_sms_deceptive||account_alert||short                             2
fraud_sms_deceptive||delivery_fee_or_service_issue||medium            1
fraud_sms_deceptive||delivery_fee_or_service_issue||short             1
fraud_sms_deceptive||prize_or_contest_scam||medium                    1
legitimate_email||administrative_communication||medium                1
legitima

,gen_id,text,label,label_str,fraudness,channel,scenario_family,subtype,target_bin,actual_bin,token_count,origin_model,split,generated_at,qc_issues,_stratum
0,4483ee6d-561d-4538-9565-b95dd7ec4a91,"I HOPE THIS MESSAGE FINDS YOU WELL,\n\nMy name...",1,llm,fraud,email,advance_fee_scam_email,diplomatic_package_or_fund_release,long,medium,358,openai/gpt-4o-mini,pilot,2026-04-08 00:59:32.456019,[],advance_fee_scam_email||diplomatic_package_or_...
1,501cc74e-8a73-4f53-8b95-458156f9d8e9,"HELLO MY DEAR FRIEND, \nI hope this message m...",1,llm,fraud,email,advance_fee_scam_email,inheritance_or_estate_transfer,medium,medium,155,openai/gpt-4o-mini,pilot,2026-04-08 00:59:13.085277,[],advance_fee_scam_email||inheritance_or_estate_...
2,a4a75b39-76a2-43b3-880c-aaee6b3ad091,"Greetings My Dearest Friend,\n\nI hope this em...",1,llm,fraud,email,advance_fee_scam_email,inheritance_or_estate_transfer,long,medium,387,openai/gpt-4o-mini,pilot,2026-04-08 00:59:04.527686,[],NaN


In [32]:
# ── Cell 2: Load human corpora ────────────────────────────────────────────
GATHERED = BASE / "data" / "interim" / "gathered"
ANNOT = BASE / "data" / "interim" / "annotated"

human = {}

# fraud email
human["phishing_email"] = pd.read_json(GATHERED / "nazario_prepared.jsonl", lines=True)
human["advance_fee_scam_email"] = pd.read_json(GATHERED / "nigerian_fraud_prepared.jsonl", lines=True)

# fraud sms
human["fraud_sms_deceptive"] = pd.concat(
    [
        pd.read_json(GATHERED / "smishtank_prepared.jsonl", lines=True),
        pd.read_json(GATHERED / "mendeley_smishing_prepared.jsonl", lines=True),
    ],
    ignore_index=True,
)

# ham email + sms (annotated)
human["legitimate_email"] = pd.concat(
    [
        pd.read_json(ANNOT / "enron_ham_annotated.jsonl", lines=True),
        pd.read_json(ANNOT / "spamassassin_ham_annotated.jsonl", lines=True),
    ],
    ignore_index=True,
)
human["legitimate_sms"] = pd.read_json(ANNOT / "sms_ham_annotated.jsonl", lines=True)

# Align ham corpora with LLM pilot schema (annotated files use business_email / personal_everyday_sms / etc.)
human["legitimate_email"] = human["legitimate_email"].copy()
human["legitimate_email"]["scenario_family"] = "legitimate_email"
human["legitimate_email"]["channel"] = "email"
human["legitimate_email"]["fraudness"] = "legitimate"

human["legitimate_sms"] = human["legitimate_sms"].copy()
human["legitimate_sms"]["scenario_family"] = "legitimate_sms"
human["legitimate_sms"]["channel"] = "sms"
human["legitimate_sms"]["fraudness"] = "legitimate"

for fam, df in human.items():
    df = df.copy()
    # ensure token_length + length_bin exist
    if "token_length" not in df.columns:
        df["token_length"] = df["text"].astype(str).str.split().str.len()
    ch = "sms" if fam.endswith("sms") else "email"
    df["channel"] = ch
    fraud_default = (
        "fraud"
        if fam in ("phishing_email", "advance_fee_scam_email", "fraud_sms_deceptive")
        else "legitimate"
    )
    df["fraudness"] = fraud_default
    if "scenario_family" not in df.columns:
        df["scenario_family"] = fam
    df["length_bin"] = df["token_length"].map(lambda n: compute_length_bin(int(n), ch))
    human[fam] = df

    print(f"{fam:22s}  rows={len(df):6d}  channel={df['channel'].mode().iloc[0]}  bins={df['length_bin'].value_counts().to_dict()}")

human["fraud_sms_deceptive"].head(2)

phishing_email          rows=  5201  channel=email  bins={'medium': 2842, 'short': 1614, 'long': 745}
advance_fee_scam_email  rows=  3234  channel=email  bins={'long': 2303, 'medium': 903, 'short': 28}
fraud_sms_deceptive     rows=   852  channel=email  bins={'short': 846, 'medium': 6}
legitimate_email        rows=   740  channel=email  bins={'medium': 334, 'short': 270, 'long': 136}
legitimate_sms          rows=   320  channel=sms  bins={'short': 248, 'medium': 69, 'long': 3}


/var/folders/22/l60tvhln5rlbvshvwggtvlc40000gn/T/ipykernel_27927/3926180009.py:24: FutureWarning: Parsed string "Fri, 20 Sep 2002 09:30:30 EDT" included an un-recognized timezone "EDT". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  pd.read_json(ANNOT / "spamassassin_ham_annotated.jsonl", lines=True),


,text,label,label_str,fraudness,channel,scenario_family,source_family,dataset_source,message_categories,sms_fraud_subtype,...,char_length,token_length,length_bin,time_band,origin_model,split,lang_filter_method,n_duplicate_rows,annotation_confidence,annotation_model
0,"Costco: Daniel, the code 42003 printed on your...",0,human,fraud,email,fraud_sms_deceptive,smishtank,Smishtank Dataset I.csv,Prize/Contest,prize_or_contest_scam,...,105,25,short,legacy,human,tbd,langdetect_v1,1,NaN,NaN
1,"Hi, you still owe UPS $4.10 USD in customs fee...",0,human,fraud,email,fraud_sms_deceptive,smishtank,Smishtank Dataset I.csv,Delivery,delivery_fee_or_service_issue,...,128,31,short,legacy,human,tbd,langdetect_v1,1,NaN,NaN


In [33]:
# ── Cell 3: Matching logic (top-2) ─────────────────────────────────────────
def pick_top2_humans(llm_row: pd.Series, human_df: pd.DataFrame) -> list[pd.Series]:
    df = human_df

    # hard constraints
    if "channel" in df.columns:
        df = df[df["channel"] == llm_row["channel"]]
    if "fraudness" in df.columns:
        df = df[df["fraudness"] == llm_row["fraudness"]]
    if "scenario_family" in df.columns:
        df = df[df["scenario_family"] == llm_row["scenario_family"]]

    if len(df) == 0:
        return []

    # prefer same subtype when human has it
    if "subtype" in df.columns and pd.notna(llm_row.get("subtype", None)):
        df_sub = df[df["subtype"] == llm_row["subtype"]]
        if len(df_sub) > 5:  # only enforce if it doesn't collapse too much
            df = df_sub

    # prefer same length bin
    if "length_bin" in df.columns and pd.notna(llm_row.get("actual_bin", None)):
        df_bin = df[df["length_bin"] == llm_row["actual_bin"]]
        if len(df_bin) > 5:
            df = df_bin

    # rank by closest token length
    target = int(llm_row["token_count"]) if pd.notna(llm_row.get("token_count", None)) else len(str(llm_row["text"]).split())
    df = df.copy()
    df["abs_len_diff"] = (df["token_length"].astype(int) - target).abs()

    df = df.sort_values(["abs_len_diff", "token_length"], ascending=[True, True])

    # pick two distinct texts
    picked = []
    seen_texts = set()
    for _, r in df.iterrows():
        t = str(r.get("text", ""))
        if not t or t in seen_texts:
            continue
        picked.append(r)
        seen_texts.add(t)
        if len(picked) == 2:
            break
    return picked


# quick smoke test on 3 rows
sample_rows = llm.sample(3, random_state=7)
for _, r in sample_rows.iterrows():
    fam = r["scenario_family"]
    matches = pick_top2_humans(r, human[fam])
    print("\n", fam, r["subtype"], r["channel"], r["fraudness"], "llm_tokens=", r["token_count"], "llm_bin=", r["actual_bin"])
    for j, m in enumerate(matches, 1):
        print(f"  human#{j}: tokens={int(m['token_length'])} bin={m.get('length_bin')} source={m.get('dataset_source', m.get('source'))}")


 legitimate_sms coordination_or_logistics sms legitimate llm_tokens= 6 llm_bin= short
  human#1: tokens=6 bin=short source=None
  human#2: tokens=6 bin=short source=None

 legitimate_email work_coordination email legitimate llm_tokens= 88 llm_bin= short
  human#1: tokens=88 bin=short source=None
  human#2: tokens=88 bin=short source=None

 fraud_sms_deceptive delivery_fee_or_service_issue sms fraud llm_tokens= 13 llm_bin= short


In [34]:
# ── Cell 4: Build markdown for ALL pilot rows ─────────────────────────────
def human_source_label(h: pd.Series, family: str) -> str:
    # gathered files store dataset_source; annotated ham may store source/dataset_source or none
    src = h.get("dataset_source")
    if pd.isna(src) or src is None:
        src = h.get("source")
    if pd.isna(src) or src is None:
        # fallback to family label
        if family == "legitimate_email":
            return "ham (enron/spamassassin annotated)"
        if family == "legitimate_sms":
            return "ham (sms annotated)"
        return "human"
    return str(src)


def md_escape(text: str) -> str:
    # We keep verbatim inside ```text blocks; nothing needed except ensure str
    return str(text)


lines = []
lines.append("# Pilot visual comparison — HUMAN vs LLM (top-2 matched, stratified subset)\n")
lines.append(f"Source of LLM texts: `{PILOT_PATH.relative_to(BASE)}`  ")
lines.append(
    f"Goal: show **{len(llm_to_match)}** stratified pilot LLM samples (target N_STRATIFIED={N_STRATIFIED}) "
    "with two closest HUMAN matches by channel/family/fraudness/subtype and length.\n"
)

lines.append("Matching heuristic used:\n")
lines.append("- **Hard match**: `channel`, `fraudness`, `scenario_family`\n")
lines.append("- **Prefer**: same `subtype` (if available in human source)\n")
lines.append("- **Prefer**: same `length_bin` (LLM `actual_bin` ↔ human `length_bin`)\n")
lines.append("- **Choose**: minimal |human token_length − llm token_count|\n")
lines.append("\n---\n")

# stable ordering: family then subtype then token_count
llm_sorted = llm_to_match.sort_values(
    ["scenario_family", "subtype", "token_count"],
    ascending=[True, True, True],
)

missing = 0
for idx, r in llm_sorted.iterrows():
    family = r["scenario_family"]
    subtype = r.get("subtype")
    channel = r.get("channel")
    fraudness = r.get("fraudness")
    llm_tokens = int(r.get("token_count"))
    llm_bin = r.get("actual_bin")

    matches = pick_top2_humans(r, human[family])
    if len(matches) < 2:
        missing += 1

    lines.append(f"## {family} — {subtype} ({channel}, {fraudness}, {llm_bin})\n")
    lines.append(f"- **LLM**: `{r.get('origin_model')}`, tokens={llm_tokens}, bin=`{llm_bin}`\n")

    # human blocks
    for j in range(2):
        if j < len(matches):
            h = matches[j]
            h_tokens = int(h.get("token_length"))
            h_bin = h.get("length_bin")
            h_src = human_source_label(h, family)
            lines.append(f"- **HUMAN #{j+1}**: {h_src}, tokens={h_tokens}, bin=`{h_bin}`\n")
        else:
            lines.append(f"- **HUMAN #{j+1}**: (no match found)\n")

    lines.append("\n### HUMAN #1\n")
    if len(matches) >= 1:
        lines.append("```text\n" + md_escape(matches[0].get("text")) + "\n```\n")
    else:
        lines.append("```text\n(NO MATCH)\n```\n")

    lines.append("\n### HUMAN #2\n")
    if len(matches) >= 2:
        lines.append("```text\n" + md_escape(matches[1].get("text")) + "\n```\n")
    else:
        lines.append("```text\n(NO MATCH)\n```\n")

    lines.append("\n### LLM\n")
    lines.append("```text\n" + md_escape(r.get("text")) + "\n```\n")
    lines.append("\n---\n")

print("Stratified LLM rows in report:", len(llm_sorted))
print("Rows with <2 human matches:", missing)
len(lines)

Stratified LLM rows in report: 20
Rows with <2 human matches: 5


229

In [35]:
# ── Cell 5: Save markdown ─────────────────────────────────────────────────
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_path = OUT_DIR / f"match_s{N_STRATIFIED}_{ts}.md"
out_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", out_path)
print("Bytes:", out_path.stat().st_size)
out_path

Saved: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-tests/match_s20_20260408_010642.md
Bytes: 46200


PosixPath('/Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-tests/match_s20_20260408_010642.md')